# 04 — Direction accuracy against effect size

**The observation.** Direction accuracy — the fraction of genes whose predicted change has the same
sign as the true change — came out at roughly 7-20% on unseen combinations in the source project.
That is not "near chance". For a binary sign call it is *far below* chance, and a number that far
below chance is information, not noise: something systematic is producing it.

**The hypothesis.** The metric is scored over all ~5,000 genes. Most of them have a true delta that
is noise around zero (see `01`), so their sign is a coin flip the model has no signal for. GEARS'
`frac_correct_direction_all` also requires an exact `sign()` match, and `sign(0)` is its own value —
so a gene whose true delta is exactly zero can never be scored correct. Add any small systematic bias
in the predicted delta and the whole population moves one way.

**The test.** Sweep `min_abs_delta` from 0 to the 99th percentile of `|true delta|` and score
direction accuracy on the surviving genes only. If the hypothesis holds, accuracy climbs toward
50-70% as the noise genes drop out.

**The rule.** The threshold grid is fixed on each run's own percentiles before any accuracy is
computed. The threshold is never chosen to improve the curve. If the curve does not climb, that is a
real and more interesting finding and it gets reported as one.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

RUNS = REPO / "runs"
TABLES = REPO / "results" / "tables"
pd.set_option("display.width", 160)

In [ ]:
from pertreadout import evaluate, figures

sweep = evaluate.direction_sweep(RUNS)
agg_sweep = evaluate.aggregate_sweep(sweep)
verdict = evaluate.sweep_verdict(agg_sweep)
verdict[verdict["model"].isin(["gears", "hgnn"])].round(4)

## The sweep

Solid lines are the mean over 5 seeds, shaded bands are 95% CIs, the dotted grey line (right axis,
log) is how many genes survive the threshold. The dashed horizontal line at 0.5 is chance for a sign
call.

The x axis is the *percentile* of `|true delta|` used as the threshold, not the threshold itself, so
the three datasets are comparable despite different effect-size scales.

In [ ]:
path = figures.fig_direction_sweep(agg_sweep, REPO / "results" / "figures")
plt.figure(figsize=(14, 4.5)); plt.imshow(plt.imread(str(path))); plt.axis("off"); plt.show()

In [ ]:
pivot = (agg_sweep[agg_sweep["model"].isin(["gears", "hgnn"])]
         .pivot_table(index=["dataset", "model"], columns="percentile",
                      values="direction_accuracy"))
cols = [c for c in pivot.columns if c in [0.0] or abs(c % 20) < 1e-9 or c > 98]
pivot[cols].round(3)

## The explanation, quantified

At `min_abs_delta = 0` every gene is scored, including the ones whose true delta is inside the
measurement noise of zero. This is the fraction of (condition, gene) pairs in that category, computed
from the standard error of each measured delta — the control mean and the condition mean are both
finite-sample averages, so their difference carries `sqrt(var_ctrl/n_ctrl + var_cond/n_cond)`.

That number is the answer to "why is a binary metric below 50%": most of what it scores is not a
binary question with a right answer.

In [ ]:
per_run = evaluate.per_run_table(RUNS)
noise = (per_run[per_run["model"].isin(["gears", "hgnn"])]
         .groupby("dataset")["frac_below_noise_floor"].agg(["mean", "min", "max"]))
print("fraction of (condition, gene) true deltas within 1.96 SE of zero:")
print(noise.round(4).to_string())

exact_zero = []
from pertreadout.train import load_predictions
for row in evaluate.find_runs(RUNS).itertuples():
    d = load_predictions(row.path)
    exact_zero.append({"dataset": row.dataset,
                       "frac_true_delta_exactly_zero": float(np.mean(d["true"] == d["ctrl_mean"]))})
print("\nfraction whose true delta is exactly zero (never scorable as correct):")
print(pd.DataFrame(exact_zero).groupby("dataset").mean().round(5).to_string())

## Reading the result

- **If accuracy climbed** with the threshold, the sub-chance number was an artifact of scoring sign on
  genes whose true delta is noise. The metric was measuring the composition of the gene panel, not the
  model.
- **If it did not climb**, the models have a systematic sign bias that survives restricting to real
  effects — a genuine defect, and a more interesting one.

Either way the fix for the metric is the same: report direction accuracy with an effect-size
threshold and state the threshold, or do not report it. And either way this is *descriptive*: it says
what the metric does on this data, not that the underlying measurement model is wrong.

The verdict table at the top of this notebook says which of the two happened on these runs; the
README repeats it.